# Rater Quality Evaluation

Measures how well each AI provider (Anthropic, OpenAI, Gemini) performed on the
subset of papers that have a known ground-truth screening decision.

Metrics computed per provider:
- **Cohen's Kappa** — agreement adjusted for chance
- **Sensitivity (Recall)** — % of true includes correctly identified
- **MCC** — Matthews Correlation Coefficient, robust to class imbalance

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    matthews_corrcoef,
    recall_score,
)

PROJECT_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").exists()
)
RESULTS_DIR = PROJECT_ROOT / "2_screening" / "results"
GROUND_TRUTH_PATH = PROJECT_ROOT / "2_screening" / "decisions" / "first.csv"
CACHE_PATH = PROJECT_ROOT / "artifacts" / "processed.csv"
ALL_PAPERS_PATH = PROJECT_ROOT / "artifacts" / "all_papers.csv"
UNANIMOUS_INCLUDE_PATH = PROJECT_ROOT / "artifacts" / "unanimous_include.csv"

In [2]:
def extract_json_from_text(text: str) -> dict:
    """Extract JSON from plain text or markdown code block.

    Falls back to a targeted regex for truncated responses that never closed
    their markdown block (e.g. model hit a length limit mid-generation).
    """
    # Happy path: complete markdown code block
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    # Plain JSON (no code fence)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Truncated response: extract decision field directly
    m = re.search(r'"decision"\s*:\s*"(include|exclude)"', text)
    if m:
        return {"decision": m.group(1)}
    raise ValueError(f"Could not extract decision from text: {text[:100]!r}")


def normalize_reason(reason) -> list:
    """Ensure reason is always a list of strings."""
    if isinstance(reason, list):
        return reason
    if isinstance(reason, str):
        return [reason]
    return []


def parse_anthropic_line(line: str) -> dict | None:
    obj = json.loads(line)
    if obj.get("result", {}).get("type") != "succeeded":
        return None
    paper_id = int(obj["custom_id"])
    content = obj["result"]["message"]["content"]
    if not content:
        return None
    parsed = extract_json_from_text(content[0]["text"])
    return {
        "paper_id": paper_id,
        "provider": "anthropic",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


def parse_openai_line(line: str) -> dict | None:
    obj = json.loads(line)
    if obj.get("error") or obj["response"]["status_code"] != 200:
        return None
    paper_id = int(obj["custom_id"])
    text = obj["response"]["body"]["output"][0]["content"][0]["text"]
    parsed = extract_json_from_text(text)
    return {
        "paper_id": paper_id,
        "provider": "openai",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


def parse_gemini_line(line: str) -> dict | None:
    obj = json.loads(line)
    paper_id = int(obj["key"])
    candidates = obj.get("response", {}).get("candidates", [])
    if not candidates:
        return None
    text = candidates[0]["content"]["parts"][0]["text"]
    parsed = extract_json_from_text(text)
    return {
        "paper_id": paper_id,
        "provider": "gemini",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


PARSERS = {
    "anthropic": parse_anthropic_line,
    "openai": parse_openai_line,
    "gemini": parse_gemini_line,
}


def load_all_results() -> pd.DataFrame:
    if CACHE_PATH.exists():
        print(f"Loading from cache: {CACHE_PATH.relative_to(PROJECT_ROOT)}")
        return pd.read_csv(CACHE_PATH)

    records = []
    for provider, parser in PARSERS.items():
        files = sorted(RESULTS_DIR.glob(f"{provider}-batch-*.jsonl"))
        for path in files:
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        record = parser(line)
                        if record:
                            records.append(record)
                    except Exception as e:
                        print(f"[{path.name}] Parse error: {e}")

    df = pd.DataFrame(records)
    df.to_csv(CACHE_PATH, index=False)
    print(f"Processed {len(df)} records -> saved to {CACHE_PATH}")
    return df


results = load_all_results()
print(f"\nRecords per provider:")
print(results.groupby("provider").size().to_string())

Loading from cache: artifacts/processed.csv

Records per provider:
provider
anthropic    3793
gemini       3795
openai       3795


In [3]:
stats = (
    results.assign(doubt=results["reason"].str.contains("doubt", na=False))
    .groupby("provider")
    .agg(
        total=("paper_id", "count"),
        include=("decision", lambda x: (x == "include").sum()),
        exclude=("decision", lambda x: (x == "exclude").sum()),
        doubt=("doubt", "sum"),
    )
    .loc[["anthropic", "gemini", "openai"]]
)
stats["include_%"] = (stats["include"] / stats["total"] * 100).round(1)
stats["doubt_%"]   = (stats["doubt"]   / stats["total"] * 100).round(1)

print("Screening statistics — full dataset\n")
print(stats.to_string())

Screening statistics — full dataset

           total  include  exclude  doubt  include_%  doubt_%
provider                                                     
anthropic   3793      601     3192    139       15.8      3.7
gemini      3795      606     3187     43       16.0      1.1
openai      3795      458     3337    209       12.1      5.5


In [4]:
truth = pd.read_csv(GROUND_TRUTH_PATH)
truth = truth.rename(columns={"id": "paper_id", "decision": "truth"})
truth["truth_bin"] = (truth["truth"] == "include").astype(int)

print(f"Ground truth: {len(truth)} papers")
print(truth["truth"].value_counts().to_string())

Ground truth: 365 papers
truth
exclude    335
include     30


In [5]:
merged = results.merge(truth[["paper_id", "truth", "truth_bin"]], on="paper_id", how="inner")
merged["decision_bin"] = (merged["decision"] == "include").astype(int)

print("Coverage (papers with ground truth matched per provider):")
print(merged.groupby("provider").size().to_string())

Coverage (papers with ground truth matched per provider):
provider
anthropic    365
gemini       365
openai       365


In [6]:
providers = ["anthropic", "openai", "gemini"]
summary_rows = []

for provider in providers:
    subset = merged[merged["provider"] == provider]
    y_true = subset["truth_bin"]
    y_pred = subset["decision_bin"]

    kappa = cohen_kappa_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"Provider: {provider.upper()}  (n={len(subset)})")
    print(f"{'='*50}")
    print(f"  Kappa:       {kappa:.4f}")
    print(f"  Sensitivity: {sensitivity:.4f}")
    print(f"  MCC:         {mcc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["exclude", "include"]))

    summary_rows.append({
        "Provider": provider,
        "N": len(subset),
        "Kappa": round(kappa, 4),
        "Sensitivity": round(sensitivity, 4),
        "MCC": round(mcc, 4),
    })


Provider: ANTHROPIC  (n=365)
  Kappa:       0.3294
  Sensitivity: 1.0000
  MCC:         0.4440

              precision    recall  f1-score   support

     exclude       1.00      0.75      0.86       335
     include       0.26      1.00      0.42        30

    accuracy                           0.77       365
   macro avg       0.63      0.87      0.64       365
weighted avg       0.94      0.77      0.82       365


Provider: OPENAI  (n=365)
  Kappa:       0.4556
  Sensitivity: 1.0000
  MCC:         0.5431

              precision    recall  f1-score   support

     exclude       1.00      0.84      0.91       335
     include       0.35      1.00      0.52        30

    accuracy                           0.85       365
   macro avg       0.68      0.92      0.72       365
weighted avg       0.95      0.85      0.88       365


Provider: GEMINI  (n=365)
  Kappa:       0.3923
  Sensitivity: 1.0000
  MCC:         0.4939

              precision    recall  f1-score   support

     e

In [7]:
summary = pd.DataFrame(summary_rows).set_index("Provider")
print("\nSummary comparison:")
print(summary.to_string())


Summary comparison:
             N   Kappa  Sensitivity     MCC
Provider                                   
anthropic  365  0.3294          1.0  0.4440
openai     365  0.4556          1.0  0.5431
gemini     365  0.3923          1.0  0.4939


## Provider agreement analysis

Papers on which all three providers agree to exclude are strong candidates for automatic exclusion,
reducing the manual review workload. The cells below quantify the agreement structure across the
full dataset and validate each tier against the ground-truth subset.

In [8]:
# Pivot to one row per paper
pivot = results.pivot(index="paper_id", columns="provider", values="decision")
pivot["n_include"] = (pivot[["anthropic", "gemini", "openai"]] == "include").sum(axis=1)

tier_labels = {0: "All 3 exclude", 1: "1/3 include", 2: "2/3 include", 3: "All 3 include"}
pivot["tier"] = pivot["n_include"].map(tier_labels)

counts = pivot["tier"].value_counts().reindex(tier_labels.values())
counts_pct = (counts / len(pivot) * 100).round(1)

agreement = pd.DataFrame({"papers": counts, "%": counts_pct})
print("Agreement tiers — full dataset\n")
print(agreement.to_string())

Agreement tiers — full dataset

               papers     %
tier                       
All 3 exclude    2974  78.4
1/3 include       284   7.5
2/3 include       230   6.1
All 3 include     307   8.1


In [9]:
# Validate each tier against the ground truth
pivot_gt = pivot.merge(truth[["paper_id", "truth"]], on="paper_id", how="inner")

rows = []
for n, label in tier_labels.items():
    sub = pivot_gt[pivot_gt["n_include"] == n]
    true_inc = (sub["truth"] == "include").sum()
    rows.append({"tier": label, "gt_papers": len(sub), "true_includes": true_inc})

validation = pd.DataFrame(rows).set_index("tier")
print("Ground truth validation\n")
print(validation.to_string())

# Summary: how many papers need review under each strategy
print("\nManual review workload by strategy\n")
for threshold, label in [(3, "All 3 include only"), (2, ">=2/3 include"), (1, "Any include (>=1/3)")]:
    n_review = (pivot["n_include"] >= threshold).sum()
    gt_sub = pivot_gt[pivot_gt["n_include"] >= threshold]
    true_inc_covered = (gt_sub["truth"] == "include").sum()
    total_true_inc = (pivot_gt["truth"] == "include").sum()
    print(f"  {label:25s}: {n_review:4d} papers to review  "
          f"({n_review/len(pivot)*100:.1f}% of total)  "
          f"| true includes covered: {true_inc_covered}/{total_true_inc}")

Ground truth validation

               gt_papers  true_includes
tier                                   
All 3 exclude        226              0
1/3 include           44              0
2/3 include           32              0
All 3 include         63             30

Manual review workload by strategy

  All 3 include only       :  307 papers to review  (8.1% of total)  | true includes covered: 30/30
  >=2/3 include            :  537 papers to review  (14.2% of total)  | true includes covered: 30/30
  Any include (>=1/3)      :  821 papers to review  (21.6% of total)  | true includes covered: 30/30


## Discussion

The ground-truth subset contains 365 papers with a notable class imbalance: 30 includes (8.2%) and 335 excludes (91.8%). Under such imbalance, accuracy is an unreliable indicator of quality — a rater that excludes every paper would reach 91.8% accuracy. Cohen's Kappa and MCC are better suited here as they account for chance agreement and class distribution respectively.

### Sensitivity

All three providers achieved perfect sensitivity (1.0) on the ground-truth subset, meaning no true include was missed. In systematic review screening this property is generally prioritised over precision, as missing a relevant paper is considered a more serious error than retaining a false positive. The trade-off is over-inclusion: each provider flagged a number of papers that a human reviewer would exclude, requiring manual resolution.

### Agreement with ground truth

| Provider  | Kappa | Sensitivity | MCC   | Specificity | Include precision |
|-----------|-------|-------------|-------|-------------|-------------------|
| OpenAI    | 0.456 | 1.000       | 0.543 | 0.84        | 0.35              |
| Gemini    | 0.392 | 1.000       | 0.494 | 0.80        | 0.31              |
| Anthropic | 0.329 | 1.000       | 0.444 | 0.75        | 0.26              |

OpenAI showed the highest agreement with the ground truth across all metrics. Its specificity of 0.84 indicates that 84% of true excludes were correctly rejected, compared to 80% for Gemini and 75% for Anthropic. Kappa values range from 0.33 (Anthropic) to 0.46 (OpenAI), corresponding to *fair* to *moderate* agreement — a range that is not unusual for automated title/abstract screening on ambiguous corpora.

MCC ranges from −1 (perfect inverse) through 0 (no better than chance) to +1 (perfect agreement). All three providers score above zero (0.44–0.54), suggesting some predictive alignment with the ground truth beyond chance. Since sensitivity is identical across all three, the differences in MCC and Kappa are driven entirely by specificity — that is, by how conservatively each provider handles exclusions. Gemini (0.494) falls between OpenAI and Anthropic on this dimension.

### Provider agreement and review workload

Cross-provider agreement provides an additional signal for prioritising manual review. In the ground-truth subset, all 30 true includes fall in the tier where all three providers agreed to include — none appear in the 1/3 or 2/3 tiers. Under this pattern, restricting manual review to papers where all three providers agree to include (307 papers, 8.1% of the full set) would recover all known true includes while avoiding review of the remaining ~91.9%. Extending the threshold to ≥2/3 agreement (537 papers, 14.2%) adds a conservative buffer at the cost of additional review effort. These estimates carry uncertainty given the limited size of the ground-truth subset.

### Limitations

These results are based on a relatively small ground-truth subset (n=365), with only 30 positive examples, which limits the precision of sensitivity and agreement estimates. The metrics should be interpreted as indicative rather than definitive.

## Export for manual review

The agreement analysis shows that, on the ground-truth subset, all 30 true includes fall in the tier where all three providers agreed to include. No true include was found among papers where only one or two providers voted to include. Based on this, the 307 papers with unanimous inclusion form the candidate set for manual full-text review.

Papers already present in the ground-truth subset have a known decision and are excluded from the export. The remaining 244 papers are exported for manual adjudication.

In [10]:
unanimous_ids = pivot[pivot["n_include"] == 3].index

papers = pd.read_csv(ALL_PAPERS_PATH, usecols=["id", "title", "abstract"])
export = (
    papers[papers["id"].isin(unanimous_ids)]
    .sort_values("id")
    [["id", "title", "abstract"]]
)

# Remove papers already decided in the ground truth
gt_ids = set(truth["paper_id"])
export = export[~export["id"].isin(gt_ids)].assign(final_decision="")

# Fill missing abstracts (manually)
empty_abstracts = {
    122: "As significant sources of energy consumption and carbon emissions, data centers have become a focal point for improving energy efficiency worldwide. To address the challenges of high computational resource demands and limited adaptability of traditional prediction models to complex conditions, this paper proposes an artificial intelligence-enabled predictive energy saving planning based on the Transformer-GRU model for predicting coolant temperature in the liquid cooling system of data centers. By integrating the self-attention mechanism of the Transformer and the time-series prediction strengths of GRU, the model performs correlation analysis and feature extraction of key parameters to achieve high-precision predictions of coolant return temperature. Experimental results demonstrate the model's superior accuracy compared to traditional prediction models, achieving an MSE of 1.349, RMSE of 1.157, MAPE of 0.0244, and R2 of 81.07 %, significantly outperforming baseline models such as Transformer-LSTM (MSE = 1.355), Informer (MSE = 1.356), Reformer (MSE = 1.353), DeepAR (MSE = 1.385), LSTM (MSE = 1.351), GRU (MSE = 1.366), and CNN-GRU (MSE = 1.363). The model maintains high predictive accuracy under fluctuating environments and complex cooling conditions, effectively reducing the operational energy consumption of the liquid cooling system. This advancement not only enhances cooling efficiency but also drives data centers toward greater intelligence and sustainability. By leveraging real-time monitoring data and predictive control, the model dynamically optimizes cooling strategies, reducing coolant and energy usage while promoting sustainable resource utilization. Additionally, this study offers implementation insights for high-performance computing environments, laying the groundwork for future research on extending model capabilities and integrating multimodal data.",
    143: "Energy consumption represents one of the most relevant issues by now in operating computing infrastructures, from traditional High Performance Computing Centers to Cloud Data Centers. Low power System-on-Chip (SoC) architectures, originally developed in the context of mobile and embedded technologies, are becoming attractive also for scientific and industrial applications given their increasing computing performances, coupled with relatively low costs and power demands. In this paper, we investigate the performance of the most representative SoCs for a computational intensive N-body benchmark, a simple deep learning based application and a real-life application taken from the field of molecular biology. The goal is to assess the trade-off among time-to-solution, energy-to-solution and economical aspects for both scientific and commercial purposes they are able to achieve in comparison to traditional server-grade architectures adopted in present infrastructures.",
    144: "The cloud computing paradigm has gained wide acceptance in the scientific community, taking a significant share from fields previously reserved exclusively for High Performance Computing (HPC). On-demand access to a large amount of computing resources provided by Cloud makes it ideal for executing large-scale optimizations using evolutionary algorithms without the need for owning any computing infrastructure. In this regard, we extended WoBinGO, an existing parallel software framework for genetic algorithm based optimization, to be used in Cloud. With these extensions, the framework is capable of elastically and frugally utilizing the underlying cloud computing infrastructure for performing computationally expensive fitness evaluations.\nWe studied two issues that are pertinent when dealing with large-scale optimization in the elastic cloud environment: the computing instance launching overhead and the price of engaging Cloud for solving optimization problems, in terms of the instances' cumulative uptime. To explain the usability limits of WoBinGO framework running in the IaaS environment, a comprehensive analysis of the framework's performance was given.\nOptimization of both total optimization time and total cumulative uptime, leads to minimizing the cost of cloud resources utilization. In this way, we are proposing an intelligent decision support engine based on artificial neural networks and metaheuristics to provide the user with an assessment of the framework's behavior on the underlying infrastructure in terms of optimization duration and the cost of resource consumption. According to a given assessment, the user can decide upon faster delivery of results or lower infrastructure costs.\nThe proposed software framework has been used to solve a complex real-world optimization problem of a subsurface rock mass model calibration. The results obtained from the private OpenStack deployment show that by using the proposed decision support engine, significant savings can be achieved in both optimization time and optimization cost.",
    145: "Very High Resolution satellite and aerial imagery are used to monitor and conduct large scale surveys of ecological systems. Convolutional Neural Networks have successfully been employed to analyze such imagery to detect large animals and salient features. As the datasets increase in volume and number of images, utilizing High Performance Computing resources becomes necessary. In this paper, we investigate three task-parallel, data-driven workflow designs to support imagery analysis pipelines with heterogeneous tasks on high performance computing platforms. We analyze the capabilities of each design when processing 3097 and 1575 images for two distinct use cases, for a total of 4,672 satellite and aerial images and 8.35 TB of data. We experimentally model the execution time of the tasks of the image processing pipelines. We perform experiments to characterize resource utilization, total time to completion and overheads of each design. Our analysis shows which design is best suited to scientific pipelines with similar characteristics.",
    149: "Combining Deep Neural Networks with Reinforcement Learning, known as Deep Reinforcement Learning (DRL), is revolutionizing fields like medicine, industry, and gaming. DRL has achieved groundbreaking results, particularly in complex Real-Time Strategy (RTS) games such as StarCraft II and Dota 2, serving as benchmarks for testing RL algorithms' robustness and safety.\nDespite these successes, DRL algorithms face challenges, including high computational costs and a lack of safety-aware approaches. Training these algorithms requires extensive computational resources, leading to a significant divide between algorithms developed on supercomputers and those feasible on standard hardware. This also raises sustainability concerns due to increased CO2 emissions. Additionally, most RL algorithms are risk-neutral, limiting their deployment in safety-critical systems.\nWe present a novel model-based DRL approach, the Safe Observations Rewards Actions Costs Learning Ensemble (S-ORACLE), to address these challenges. S-ORACLE balances robust safety awareness with minimized risk and computational efficiency. Empirical validation across complex game environments—Deep RTS, ELF: MiniRTS, MicroRTS, Deep Warehouse, and StarCraft II—demonstrates that S-ORACLE outperforms state-of-the-art methods by significantly improving safety performance, reducing computational costs, and lowering environmental impact, while maintaining high efficiency and adaptability in training.",
    166: "Data Centres (DCs), the core of the ever-increasing economic and societal activities, are experiencing high energy consumption due to the rapidly growing demand for digital services, which leads to the largest DCs' operational costs and significant environmental and power security impacts. Hence, optimising their energy efficiency is a critical and top priority to ensure economic and environmentally sustainable DC management. However, prior heuristics and engineering-based solutions are inadequate due to the increasing physical complexity and sheer number of possible configurations, non-linear system interactions, and the growing monitoring of operational management data. Therefore, this paper develops a six-layered hybrid Convolutional Neural Network (CNN) and Long-Short-Term Memory (LSTM), called CNN-LSTM, a suitable data-driven Deep Learning (DL) model inspired by the human brain for effectively modelling the complex and plant performance and optimising DC efficiency. The proposed model uses CNN to capture complex parameter interactions and spatial pattern recognition and LSTM to capture temporal dependencies of the time series operational data to predict the next energy consumption value and optimise it using a cooling system fan speed controllable variable. The model was extensively trained and tested using actual operational management data obtained from the Enea High-Performance Computing (HPC) CRESCO6 cluster, Italy. 80% of the data was used for training, and the rest 20% for testing. According to the experimental results, it accurately predicts energy consumption every 15 minutes with an average Mean Absolute Error (MAE) of 0.0043. A sensitivity analysis optimisation strategy is also implemented using various cooling system fan speed set points. When the fan speed optimal set-point is automatically reduced by 50%, the energy consumption is optimised by an average of 0.0029 kWh with a 0.0039 MAE over the last 15 minutes of the testing set. Hence, this paper found that the proposed CNN-LSTM deep learning predictive model can effectively mimic actual DC operations and optimise efficiency. By simulating this model, DC operators can effectively manage and optimise their DC energy efficiency while reducing energy and environmental costs",
    167: "The rapid advancements in AI and Machine Learning necessitate a robust computational infrastructure to support cutting-edge research and industrial applications. From the academic and industrial AI community perspective, voiced in the recent ELISE project, the European AI platform is recommended to center around the EuroHPC growing ecosystem. It should be user-driven, easily accessible, powerful, and compliant with European regulations. AI-optimized and dedicated supercomputers for the European AI community are also coming, in addition to upgrading partitions of existing EuroHPC systems to 'AI enabled' stage. Related calls have been initiated in September 2024. Further, conventional EuroHPC systems are suggested to be extended with quantum computing, edge AI, and neuromorphic computing to cater to AI models deployed on network edge devices and sustainability in the long run. The challenges are presented in three case studies, ranging from training Transformers on HPC to LLMs trained federally across three different Euro HPC systems to recent results on hybrid classical-quantum application. This paper concludes with case studies results-informed next steps believed to benefit AI practitioners and the broader AI community.",
    188: "The necessity for capping carbon emission has significantly restricted the potential of modern data centers. For this matter, both industry and academia are proactively seeking opportunities on cross-layer power management schemes that could open a door for sustainable high-performance computing platform. In this paper we investigate an emerging trend in the IT industry: using promising onsite distributed generation (DG) techniques to provide premium clean energy to the computing load. We develop data center power demand shaping (PDS), a novel technique that allows data centers to utilize onsite green energy efficiently. In contrast to prior design, PDS takes advantage of a so-far unexplored power supply feature, i.e., the load following capabilities of DG systems to avoid the high performance penalty issue incurred during supply tracking. In addition, PDS features two adaptive power management schemes: DGR Boost and UPS Boost. These two workload-aware optimization methods leverage mature computer tuning knobs to achieve attractive data center performance improvement. Using real-world data center traces and industry data of distributed generation systems, we show that our technique can come within 1.2\\% performance of an ideal oracle, which is roughly a 37\\% improvement over existing supply tracking based design. Our design could save over 100 metric tons of carbon emissions annually for a 10MW data center.",
    230: "Presents the title page of the proceedings record.",
    263: "Effective solving complex mathematical modeling problems is based on the use of high-performance computing. Clouds, grids, and public access supercomputer centers are commonly used platforms. Their integration into a unified environment provides possibilities for carrying out mass large-scale scientific experiments and efficient scalable resource allocation at different stages of the application design and execution. However, end-users have to carefully select optimization criteria such as completion time, deadlines, reliability, cost, etc. It is a complicated problem due to integrated resources differ significantly in their computing capabilities, hardware and software platforms, system architectures, user interfaces, etc. The paper presents new features of the Orlando Tools framework for the development of distributed applied software packages (scalable scientific applications) that mitigates various types of uncertainties arising from the job distribution in the integrated computing environment. It provides continuous integration, delivery, and deployment of applied and system software to significantly mitigate the negative impact of uncertainty on problem-solving time, computation reliability, and resource efficiency. An experimental analysis of the sustainable design and development of the real energy sector clearly demonstrates the advantages of the tools.",
    271: "No abstract available.",
    582: "No abstract available.",
    845: "At the National Renewable Energy Laboratory (NREL)?a U.S. Department of Energy laboratory?computational science, high-performance computing, applied mathematics, advanced computer science, visualization, and data play a pivotal role in advancing energy abundance, affordability, security, and reliability. From fundamental scientifc discovery to systems engineering and analysis, NREL researchers tackle market-relevant challenges to develop solutions for an independent energy system that is reliable, resilient and secure. Collaborative partnerships with industry, government, and academia ensure that our research remains cutting edge, impactful, applicable, and aligned with real-world energy needs. This special issue of Computing in Science & Engineering highlights exemplary NREL projects where computational tools and methodologies drive discovery and accelerate innovation in scalable and integrated energy systems. The featured articles explore the role of computational modeling, high-performance computing, generative AI, and adaptive computing in advancing independent energy solutions, optimizing sustainability research, and enhancing decision-making for energy solutions using a broad mix of energy technologies. Here, these contributions demonstrate how NREL?s computational research bridges the gap between theoretical advancements and practical implementation, emphasizing interdisciplinary collaboration and a commitment to innovation, with a focus on translating computational excellence into real-world impact, thus accelerate progress toward national energy goals. By showcasing cutting-edge research at the intersection of computational science and energy systems, this issue aims to inspire and inform researchers, practitioners, and policymakers dedicated to shaping a more reliable energy future.",
    2118: "No abstract available.",
    2405: "No abstract available.",
    2570: "No abstract available.",
    2696: "No abstract available.",
    3173: "No abstract available.",
    3241: "No abstract available.",
    3242: "No abstract available.",
    3642: "No abstract available.",
}

filled = export["id"].map(empty_abstracts)
missing_before = export["abstract"].isna().sum()
export["abstract"] = export["abstract"].fillna(filled)
missing_after = export["abstract"].isna().sum()

export_path = UNANIMOUS_INCLUDE_PATH
export.to_csv(export_path, index=False)
print(f"Exported {len(export)} papers -> {export_path.relative_to(PROJECT_ROOT)}")
print(f"  Abstracts filled from mapping: {missing_before - missing_after}  |  still missing: {missing_after}")

Exported 244 papers -> artifacts/unanimous_include.csv
  Abstracts filled from mapping: 20  |  still missing: 0
